In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import tempfile
import importlib.util
from types import SimpleNamespace

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# =========================
# EDIT THESE
# =========================

CODE_PATH = Path("/vol/biomedic3/tx1215/mamo-flow/src/data_handle/write_latent_cache.py")  # change this to your .py path if needed

SPLIT_DIR = Path("/path/to/split_dir")        # should contain train.csv, valid.csv, test.csv
DATA_DIR = Path("/path/to/image_root")        # root folder for relative image_path values

OUT_DIR = Path("./latent_cache_test_out")

IMG_HEIGHT = 256
IMG_WIDTH = 192
BATCH_SIZE = 4
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Optional real VAE test
RUN_REAL_VAE_TEST = False
VAE_CKPT = "flux2"  # or "/path/to/local_vae_checkpoint.pt"

# Optional tiny memmap write test using fake VAE
RUN_FAKE_WRITE_TEST = True

In [2]:
assert CODE_PATH.exists(), f"CODE_PATH does not exist: {CODE_PATH}"

spec = importlib.util.spec_from_file_location("latent_cache_script", CODE_PATH)
latent_script = importlib.util.module_from_spec(spec)
spec.loader.exec_module(latent_script)

print("Imported script successfully.")
print("Available functions/classes:")
for name in [
    "preprocess_breast",
    "SplitImageDataset",
    "load_vae",
    "encode_with_vae",
    "RunningChannelStats",
    "load_split_csvs",
    "write_split_latents",
]:
    print(f"  {name}:", hasattr(latent_script, name))

AssertionError: CODE_PATH does not exist: /mnt/data/Pasted code.py

In [ ]:
split_csvs = latent_script.load_split_csvs(SPLIT_DIR)

for split, path in split_csvs.items():
    df = pd.read_csv(path, low_memory=False)
    print("=" * 80)
    print(split, path)
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print(df.head())

    required = {"cache_idx", "image_path"}
    missing = required - set(df.columns)
    print("missing required columns:", missing)

    if "cache_idx" in df.columns:
        cache_idx = df["cache_idx"].to_numpy()
        print("cache_idx dtype:", cache_idx.dtype)
        print("cache_idx min/max:", cache_idx.min(), cache_idx.max())
        print("cache_idx unique:", len(np.unique(cache_idx)), "/", len(cache_idx))
        print("cache_idx sorted equals 0..N-1:",
              np.array_equal(np.sort(cache_idx.astype(np.int64)), np.arange(len(df))))

In [ ]:
def resolve_image_path(image_path, data_dir):
    image_path = Path(str(image_path))
    if not image_path.is_absolute():
        image_path = Path(data_dir) / image_path
    return image_path

for split, path in split_csvs.items():
    df = pd.read_csv(path, low_memory=False)
    resolved = [resolve_image_path(p, DATA_DIR) for p in df["image_path"].head(20)]
    exists = [p.exists() for p in resolved]

    print("=" * 80)
    print(split)
    for p, ok in zip(resolved[:10], exists[:10]):
        print(ok, p)

    print(f"exists in first 20: {sum(exists)}/{len(exists)}")

In [ ]:
split = "train"
df = pd.read_csv(split_csvs[split], low_memory=False)

sample_paths = [
    resolve_image_path(p, DATA_DIR)
    for p in df["image_path"].head(6)
]

for p in sample_paths:
    assert p.exists(), f"Missing image: {p}"

processed = []

for p in sample_paths:
    img = latent_script.preprocess_breast(p)
    processed.append(img)

    print("=" * 80)
    print("path:", p)
    print("shape:", img.shape)
    print("dtype:", img.dtype)
    print("min/max:", img.min(), img.max())
    print("mean/std:", float(img.mean()), float(img.std()))
    print("nonzero fraction:", float((img > 0).mean()))

In [ ]:
n = len(processed)
plt.figure(figsize=(4 * n, 4))

for i, img in enumerate(processed):
    plt.subplot(1, n, i + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"{i}\n{img.shape}\n{img.dtype}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for p in sample_paths[:3]:
    default = cv2.imread(str(p))
    unchanged = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)

    print("=" * 80)
    print("path:", p)

    if default is None:
        print("cv2.imread default failed")
    else:
        print("default shape/dtype/min/max:",
              default.shape, default.dtype, default.min(), default.max())

    if unchanged is None:
        print("cv2.imread unchanged failed")
    else:
        print("unchanged shape/dtype/min/max:",
              unchanged.shape, unchanged.dtype, unchanged.min(), unchanged.max())

    if unchanged is not None and unchanged.dtype != np.uint8:
        print("WARNING: source image is not uint8. Default cv2.imread() may alter intensity.")

In [ ]:
dataset = latent_script.SplitImageDataset(
    split_csv=split_csvs["train"],
    data_dir=DATA_DIR,
    img_height=IMG_HEIGHT,
    img_width=IMG_WIDTH,
)

print("dataset length:", len(dataset))

item = dataset[0]
print("item keys:", item.keys())
print("x shape:", item["x"].shape)
print("x dtype:", item["x"].dtype)
print("x min/max:", item["x"].min().item(), item["x"].max().item())
print("cache_idx:", item["cache_idx"])
print("shortpath:", item["shortpath"])

In [ ]:
x = item["x"]

plt.figure(figsize=(4, 4))
plt.imshow(x.squeeze(0).numpy(), cmap="gray")
plt.title(f"Transformed tensor\nshape={tuple(x.shape)}, range=({x.min():.3f},{x.max():.3f})")
plt.axis("off")
plt.show()

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.startswith("cuda"),
    drop_last=False,
)

batch = next(iter(loader))

print("batch keys:", batch.keys())
print("x:", batch["x"].shape, batch["x"].dtype, batch["x"].min().item(), batch["x"].max().item())
print("cache_idx:", batch["cache_idx"])
print("shortpath example:", batch["shortpath"][0])

# Convert to VAE input range
x_vae = batch["x"].to(DEVICE) * 2.0 - 1.0

print("x_vae:", x_vae.shape, x_vae.dtype, x_vae.min().item(), x_vae.max().item())

In [ ]:
class FakeDeterministicVAE(torch.nn.Module):
    """
    Tiny fake VAE:
    - accepts [B,1,H,W]
    - returns [B,4,H/8,W/8]
    """
    beta = 0.0

    def encode(self, x, sample=False):
        z = torch.nn.functional.interpolate(
            x,
            scale_factor=1 / 8,
            mode="bilinear",
            align_corners=False,
        )
        z = z.repeat(1, 4, 1, 1)
        return z

fake_vae = FakeDeterministicVAE().to(DEVICE).eval()

with torch.inference_mode():
    z = latent_script.encode_with_vae(
        fake_vae,
        x_vae,
        sample_posterior=False,
    )

print("fake z shape:", z.shape)
print("fake z dtype:", z.dtype)
print("fake z min/max:", z.min().item(), z.max().item())
assert z.ndim == 4

In [ ]:
stats = latent_script.RunningChannelStats(channels=z.shape[1])
stats.update(z)
meta = stats.finalize()

print(json.dumps(meta, indent=2)[:1000])

In [ ]:
if RUN_FAKE_WRITE_TEST:
    tmp_dir = Path(tempfile.mkdtemp(prefix="latent_cache_fake_test_"))
    tiny_split_dir = tmp_dir / "splits"
    tiny_out_dir = tmp_dir / "out"
    tiny_split_dir.mkdir(parents=True, exist_ok=True)
    tiny_out_dir.mkdir(parents=True, exist_ok=True)

    print("tmp_dir:", tmp_dir)

    # Make tiny train/valid/test CSVs with local cache_idx 0..N-1
    for split_name, csv_path in split_csvs.items():
        df = pd.read_csv(csv_path, low_memory=False).head(8).copy()
        df["cache_idx"] = np.arange(len(df), dtype=np.int64)
        df.to_csv(tiny_split_dir / f"{split_name}.csv", index=False)

    args = SimpleNamespace(
        data_dir=str(DATA_DIR),
        out_dir=str(tiny_out_dir),
        img_height=IMG_HEIGHT,
        img_width=IMG_WIDTH,
        batch_size=2,
        num_workers=0,
        device=DEVICE,
        sample_posterior=0,
        file_prefix="fake_encoding_float32",
        overwrite=1,
        vae_ckpt="fake",
    )

    result = latent_script.write_split_latents(
        split="train",
        split_csv=tiny_split_dir / "train.csv",
        args=args,
        vae=fake_vae,
    )

    print(json.dumps(result, indent=2))

    dat_path = Path(result["file"])
    assert dat_path.exists(), dat_path

    shape = (result["num_samples"], *result["latent_shape"])
    mmap = np.memmap(dat_path, mode="r", dtype=np.float32, shape=shape)

    print("memmap shape:", mmap.shape)
    print("memmap dtype:", mmap.dtype)
    print("memmap min/max:", float(mmap.min()), float(mmap.max()))

    # Visualize first latent channel
    plt.figure(figsize=(4, 4))
    plt.imshow(mmap[0, 0], cmap="gray")
    plt.title("First fake latent channel")
    plt.axis("off")
    plt.show()
else:
    print("Skipping fake write test.")